# LLaVA — Ablation study

| Condición | Input |
|-----------|-----------|
| **FaceOnly** | Face crop (Haar cascade) |
| **FaceScene** | Full image + face crop|

## 1. Imports

In [ ]:
import warnings
from pathlib import Path
from typing import Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm
import torch
from sklearn.metrics import (
    confusion_matrix, f1_score, accuracy_score,
)

from llava_shared import load_llava_model, parse_emotion, MAX_NEW_TOKENS

warnings.filterwarnings("ignore")
print(f"CUDA avaiable: {torch.cuda.is_available()}")

## 2. Config

In [ ]:
NCAERS_ROOT = Path("datasets/NCAERS")
FACE_CROPS_ROOT = Path("datasets/NCAERS_faces")
FACE_CROPS_INDEX = FACE_CROPS_ROOT / "ncaers_faces_index.parquet"

BASELINE_PARQUET = Path("ncaers_llava_zeroshot_v2_test.parquet")
OUT_FACE_ONLY = Path("ncaers_llava_face_only_test.parquet")
OUT_FACE_SCENE = Path("ncaers_llava_face_scene_test.parquet")

EMOTIONS = ["Anger", "Disgust", "Fear", "Happy", "Neutral", "Sad", "Surprise"]
EMOTIONS_STR = ", ".join(EMOTIONS)

SAMPLE_N = None
RANDOM_SEED = 42

FOLDER_MAP = {"train": "train", "val": "validation", "test": "test", "validation": "validation"}

## 3. Dataset load

In [ ]:
if not FACE_CROPS_INDEX.exists():
    raise FileNotFoundError(
        f"{FACE_CROPS_INDEX} not found."
    )

df_faces = pd.read_parquet(FACE_CROPS_INDEX)
df_test  = df_faces[df_faces["split"] == "test"].reset_index(drop=True)

if SAMPLE_N is not None:
    df_test = df_test.sample(n=min(SAMPLE_N, len(df_test)),
                             random_state=RANDOM_SEED).reset_index(drop=True)

print(f"Test cases: {len(df_test):,}")
print(f"Face detected: {df_test["face_detected"].mean()*100:.2f}%")
print()
print(df_test["label"].value_counts().to_string())

## 4. Model LLaVA load

In [ ]:
model, processor, device = load_llava_model()

### Prompts & inference functions

In [ ]:
PROMPT_FACE_ONLY = (
    f"USER: <image>\n"
    f"Look carefully at the facial expression in this image. "
    f"Identify the primary emotion of the person. "
    f"Choose exactly one from: {EMOTIONS_STR}. "
    f"Reply with only the emotion name, nothing else.\n"
    f"ASSISTANT:"
)

PROMPT_FACE_SCENE = (
    f"USER: <image><image>\n"
    f"The first image is a close-up of the person"s face. "
    f"The second image shows the full scene. "
    f"Using both the facial expression and the contextual scene, "
    f"identify the primary emotion. "
    f"Choose exactly one from: {EMOTIONS_STR}. "
    f"Reply with only the emotion name, nothing else.\n"
    f"ASSISTANT:"
)


@torch.inference_mode()
def classify_face_only(face_img: Image.Image) -> Tuple[Optional[str], str]:
    inputs = processor(text=PROMPT_FACE_ONLY, images=face_img, return_tensors="pt").to(device)
    out    = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    raw    = processor.decode(out[0], skip_special_tokens=True)
    return parse_emotion(raw, EMOTIONS), raw


@torch.inference_mode()
def classify_face_and_scene(face_img: Image.Image, full_img: Image.Image) -> Tuple[Optional[str], str]:
    inputs = processor(text=PROMPT_FACE_SCENE, images=[face_img, full_img], return_tensors="pt").to(device)
    out    = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    raw    = processor.decode(out[0], skip_special_tokens=True)
    return parse_emotion(raw, EMOTIONS), raw


def load_images(row: pd.Series) -> Tuple[Optional[Image.Image], Optional[Image.Image]]:
    face_path = FACE_CROPS_ROOT / row["split"] / row["label"] / row["filename"]
    full_path = NCAERS_ROOT / FOLDER_MAP[row["split"]] / row["label"] / row["filename"]
    face_img  = Image.open(face_path).convert("RGB") if face_path.exists() else None
    full_img  = Image.open(full_path).convert("RGB") if full_path.exists() else None
    return face_img, full_img

print("Funciones de inferencia definidas.")

## 5. Face only

### Inference

In [ ]:
if OUT_FACE_ONLY.exists():
    print(f"Loading results from {OUT_FACE_ONLY}...")
    df_face_only = pd.read_parquet(OUT_FACE_ONLY)
    print(f"{len(df_face_only):,} cases loaded.")
else:
    results, errors = [], []
    for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc="FaceOnly"):
        try:
            face_img, _ = load_images(row)
            if face_img is None:
                errors.append({**row.to_dict(), "error": "face_not_found"})
                continue
            predicted, raw = classify_face_only(face_img)
            results.append({
                "split": row["split"], "label": row["label"], "filename": row["filename"],
                "face_detected": row["face_detected"],
                "true_emotion": row["label"], "predicted_emotion": predicted,
                "raw_response": raw,
                "correct": predicted == row["label"] if predicted else False,
                "unparseable": predicted is None,
            })
        except Exception as e:
            errors.append({**row.to_dict(), "error": str(e)})

    df_face_only = pd.DataFrame(results)
    df_face_only.to_parquet(OUT_FACE_ONLY, index=False)
    print(f"Procesesd: {len(df_face_only):,} | Errors: {len(errors):,}")
    print(f"Saved in {OUT_FACE_ONLY}")

### Evaluation

In [ ]:
def evaluate_condition(df: pd.DataFrame, tag: str, save_stem: str) -> dict:
    df_ev = df[~df["unparseable"]].copy()
    n_unp = df["unparseable"].sum()

    acc = accuracy_score(df_ev["true_emotion"], df_ev["predicted_emotion"])
    mf1 = f1_score(df_ev["true_emotion"], df_ev["predicted_emotion"],
                      labels=EMOTIONS, average="macro", zero_division=0)
    f1_per = f1_score(df_ev["true_emotion"], df_ev["predicted_emotion"],
                      labels=EMOTIONS, average=None, zero_division=0)

    cm = confusion_matrix(df_ev["true_emotion"], df_ev["predicted_emotion"], labels=EMOTIONS)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    acc_per = np.diag(cm_norm)
    mean_acc = acc_per.mean()

    print(f"Accuracy: {acc:.3f} ({acc*100:.1f}%)")
    print(f"Mean acc: {mean_acc:.3f}")
    print(f"Macro F1: {mf1:.3f}")
    for emo, a, f in zip(EMOTIONS, acc_per, f1_per):
        print(f"  {emo:<12} acc={a:.3f}  F1={f:.3f}")
    print(f"  Without parsing: {n_unp} ({n_unp/len(df)*100:.1f}%)")

    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=EMOTIONS, yticklabels=EMOTIONS,
                linewidths=0.5, ax=ax)
    ax.set_xlabel("Predicción"); ax.set_ylabel("Real")
    ax.set_title(f"LLaVA — {tag}  |  acc={acc:.3f}  F1={mf1:.3f}")
    plt.xticks(rotation=30, ha="right"); plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(f"{save_stem}_confusion_counts.png", dpi=150, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=EMOTIONS, yticklabels=EMOTIONS,
                linewidths=0.5, ax=ax)
    ax.set_xlabel("Predicción"); ax.set_ylabel("Real")
    ax.set_title(f"LLaVA — {tag} (Recall)")
    plt.xticks(rotation=30, ha="right"); plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(f"{save_stem}_confusion_norm.png", dpi=150, bbox_inches="tight")
    plt.show()

    return {
        "tag": tag, "n": len(df_ev), "accuracy": acc, "mean_acc": mean_acc,
        "f1_macro": mf1,
        "f1_per_class": dict(zip(EMOTIONS, f1_per.tolist())),
        "acc_per_class": dict(zip(EMOTIONS, acc_per.tolist())),
        "unparseable_pct": n_unp / len(df) * 100,
    }


metrics_face_only = evaluate_condition(df_face_only, "Face only", "llava_ncaers_face_only")

## 6. Full image + face crop

### Inference

In [ ]:
if OUT_FACE_SCENE.exists():
    print(f"Loading results from {OUT_FACE_SCENE}...")
    df_face_scene = pd.read_parquet(OUT_FACE_SCENE)
    print(f"{len(df_face_scene):,} cases loaded.")
else:
    results, errors = [], []
    for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc="FaceScene"):
        try:
            face_img, full_img = load_images(row)
            if face_img is None or full_img is None:
                errors.append({**row.to_dict(),
                                "error": f"missing: face={face_img is None} full={full_img is None}"})
                continue
            predicted, raw = classify_face_and_scene(face_img, full_img)
            results.append({
                "split": row["split"], "label": row["label"], "filename": row["filename"],
                "face_detected": row["face_detected"],
                "true_emotion": row["label"], "predicted_emotion": predicted,
                "raw_response": raw,
                "correct": predicted == row["label"] if predicted else False,
                "unparseable": predicted is None,
            })
        except Exception as e:
            errors.append({**row.to_dict(), "error": str(e)})

    df_face_scene = pd.DataFrame(results)
    df_face_scene.to_parquet(OUT_FACE_SCENE, index=False)
    print(f"Procesed: {len(df_face_scene):,} | Errors: {len(errors):,}")
    print(f"Saved in {OUT_FACE_SCENE}")

### Evaluation

In [ ]:
metrics_face_scene = evaluate_condition(df_face_scene, "Face + Full Img", "llava_ncaers_face_scene")